# DDSP Synthesis: Extract Audio Parameters from Real .wav

In this notebook, we:
- Load and normalize input audio files
- Trim or pad them to 4 seconds at 16kHz
- Extract pitch (f₀) and confidence using the CREPE model
- Extract loudness (in dB) using DDSP utilities
- Interpolate all sequences to 1000 time frames
- Save the extracted features into `.npz` files for later use
- Plot and store visualizations of all audio features

In [6]:
# Setup & Imports
import os
import numpy as np
import librosa
import soundfile as sf
import tensorflow as tf
import matplotlib.pyplot as plt
from tqdm import tqdm
from ddsp import spectral_ops, processors


In [7]:
# Configuration
SAMPLE_RATE = 16000
DURATION = 4.0
N_SAMPLES = int(SAMPLE_RATE * DURATION)  # 64000
N_FRAMES = 1000                          # Frame length after interpolation
AUDIO_DIR = '../data/audio_references'
OUTPUT_DIR = '../output/audio_features'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
#Linear Interpolation
def linear_interpolate(array, target_length=N_FRAMES):
    x_old = np.linspace(0, 1, num=len(array))
    x_new = np.linspace(0, 1, num=target_length)
    return np.interp(x_new, x_old, array)

In [9]:
import crepe

def compute_f0_crepe_standalone(audio, sample_rate=SAMPLE_RATE):
    # Ensure audio is in float32
    audio = audio.astype(np.float32)

    # Predict pitch using CREPE
    time, frequency, confidence, activation = crepe.predict(
        audio,
        sample_rate,
        viterbi=True,
        step_size=10  # ms between frames → ~100 fps
    )

    return frequency, confidence

#Feature Extraction Function
def extract_features(audio_path):
    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    audio = librosa.util.normalize(audio)

    # Pad or trim to 4 seconds
    if len(audio) < N_SAMPLES:
        audio = np.pad(audio, (0, N_SAMPLES - len(audio)), mode='constant')
    else:
        audio = audio[:N_SAMPLES]

    # Loudness via DDSP
    loudness_db = spectral_ops.compute_loudness(audio, SAMPLE_RATE)

    # F0 with CREPE (standalone)
    try:
        f0_hz, f0_confidence = compute_f0_crepe_standalone(audio)
    except Exception as e:
        print(f"⚠️ CREPE fallback on {os.path.basename(audio_path)}: {e}")
        f0_hz = np.zeros(N_FRAMES)
        f0_confidence = np.zeros(N_FRAMES)

    # Interpolate all features to match N_FRAMES
    f0 = linear_interpolate(f0_hz, target_length=N_FRAMES)
    conf = linear_interpolate(f0_confidence, target_length=N_FRAMES)
    loud = linear_interpolate(loudness_db, target_length=N_FRAMES)
    audio_interp = linear_interpolate(audio, target_length=N_FRAMES)

    return {
        'f0_hz': f0,
        'f0_confidence': conf,
        'loudness_db': loud,
        'audio': audio_interp
    }


In [10]:
#Processing Loop & Saving
for fname in tqdm(os.listdir(AUDIO_DIR)):
    if not fname.endswith('.wav'):
        continue

    path = os.path.join(AUDIO_DIR, fname)
    try:
        features = extract_features(path)
        name = os.path.splitext(fname)[0]
        np.savez_compressed(os.path.join(OUTPUT_DIR, f"{name}.npz"), **features)
        print(f"Saved: {name}.npz")
    except Exception as e:
        print(f"Error processing {fname}: {e}")



  0%|          | 0/10 [00:00<?, ?it/s]

13/13 [==============================] - 4s 233ms/step


 10%|█         | 1/10 [00:06<00:57,  6.38s/it]

Saved: a_baby_crying.npz


c:\Users\ADV\Universita\Magistrale\DL\awol-audio\awol-ddsp-env\lib\site-packages\librosa\core\convert.py:1332: RuntimeWarning: divide by zero encountered in log10
  + 2 * np.log10(f_sq)


...

Saved: ocean_waves_crashing.npz


c:\Users\ADV\Universita\Magistrale\DL\awol-audio\awol-ddsp-env\lib\site-packages\librosa\core\convert.py:1332: RuntimeWarning: divide by zero encountered in log10
  + 2 * np.log10(f_sq)


13/13 [==============================] - 3s 239ms/step


100%|██████████| 10/10 [01:04<00:00,  6.43s/it]

Saved: people_laughing_loudly.npz


In [ ]:
#Plotting the parameters
PLOTS_DIR = '../output/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

def plot_and_save_features(npz_path, save_dir):
    """Plot f0, loudness, and audio waveform and save as PNG."""
    data = np.load(npz_path)
    name = os.path.splitext(os.path.basename(npz_path))[0]
    plt.figure(figsize=(12, 6))
    plt.suptitle(name, fontsize=14)

    plt.subplot(3, 1, 1)
    plt.plot(data['f0_hz'])
    plt.title('f0_hz')

    plt.subplot(3, 1, 2)
    plt.plot(data['loudness_db'])
    plt.title('loudness_db')

    plt.subplot(3, 1, 3)
    plt.plot(data['audio'])
    plt.title('audio waveform')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    save_path = os.path.join(save_dir, f"{name}.png")
    plt.savefig(save_path)
    plt.close()

# Generate and save all plots
npz_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.npz')]
for fname in npz_files:
    full_path = os.path.join(OUTPUT_DIR, fname)
    plot_and_save_features(full_path, PLOTS_DIR)

print(f"Saved {len(npz_files)} plots to: {PLOTS_DIR}")



Saved 10 plots to: ../output/plots
